In [1]:
from z3 import *

def get_new_vars(vars,suffix):
    #new_vars = [Const(var.sexpr()+suffix,var.sort()) for var in vars]
    new_vars = [FreshConst(var.sort()) for var in vars]
    var_map = [(vars[i],new_vars[i]) for i in range(len(new_vars))]
    return new_vars,var_map

def Equal(a,b):
    if isinstance(a, list) and isinstance(b,list):
        if len(a)==len(b):
            return And([a[i]==b[i] for i in range(len(b))])
        else:
            return False
        
def Only(vars,cond):
    new_vars,var_map = get_new_vars(vars,"_Only")
    new_cond=substitute(cond,*var_map)
    return And(cond,
               Not(Exists(new_vars,And(new_cond,
                                   Not(Equal(vars,new_vars))))))
def ExistsOnly(vars,cond):
    return Exists(vars,Only(vars,cond))

def OneOfMany(vars,cond):
    new_vars,var_map = get_new_vars(vars,"_OneOfMany")
    new_cond=substitute(cond,*var_map)
    return And(cond,ExistsOnly(new_vars,And(new_cond,Not(Equal(new_vars,vars)))))

# def ExistsMany(vars,cond):
#     return Exists(vars,OneOfMany(vars,cond))

def WithoutKnowingCanNotDetermine(vars,known_conds,res):#if vars are unknown and known_conds are known then we can definitely determnie res
    new_vars,var_map = get_new_vars(vars,"_WithoutKnowingCanNotDetermine")
    new_cond=substitute(known_conds,*var_map)
    new_res=substitute(res,*var_map)
    return Exists(vars,And(known_conds,Exists(new_vars,And(new_cond,new_res!=res,Not(Equal(vars,new_vars))))))

def WithoutKnowingCanDetermine(vars,known_conds,res):
    new_vars,var_map = get_new_vars(vars,"_WithoutKnowingCanDetermine")
    new_cond=substitute(known_conds,*var_map)
    new_res=substitute(res,*var_map)
    return Exists(vars,And(known_conds,Not(Exists(new_vars,And(new_cond,new_res!=res,Not(Equal(vars,new_vars)))))))

#TODO implement KnowingOnlyCanNotDetermine function which is same as WithoutKnowingCanNotDetermine but we assume that we only know vars and other vars from known_conds are unknown,only we should be able to give not only vars that known but also exprs from vars(to do that we can make new var and add new cond that newly created var equal to expr)

def ExistsMany(vars,cond):
    new_vars,var_map = get_new_vars(vars,"_ExistsMany")
    new_cond=substitute(cond,*var_map)
    return Exists(vars+new_vars,And(new_cond,cond,Not(Equal(new_vars,vars))))

def ExistsN(n,vars,cond):
    if isinstance(n,int):
        solutions = [get_new_vars(vars,"_ExistsN_%i" % i) for i in range(n)]
        all_vars = sum([s[0] for s in solutions],[])
        all_conds = And([substitute(cond,*s[1]) for s in solutions])
        return Exists(all_vars,And(all_conds,Distinct(all_vars),Not(Exists(vars,And(cond,And([Not(Equal(s[0],vars)) for s in solutions]))))))

def OnlyGives(vars,cond,res):
    new_vars,var_map = get_new_vars(vars,"_OnlyGives")
    new_cond=substitute(cond,*var_map)
    new_res=substitute(res,*var_map)
    return And(cond,Not(Exists(new_vars,And(new_cond,new_res==res,Not(Equal(new_vars,vars))))))

def OneOfManyGives(vars,cond,res):
    new_vars,var_map = get_new_vars(vars,"_OneOfManyGives")
    new_cond=substitute(cond,*var_map)
    new_res=substitute(res,*var_map)
    return And(cond,Exists(new_vars,And(new_cond,new_res==res,Not(Equal(new_vars,vars)))))

def ExistsOnlyGives(vars,cond,res):
    return Exists(vars,OnlyGives(vars,cond,res))

# def ExistsManyGives(vars,cond,res):
#     return Exists(vars,OneOfManyGives(vars,cond,res))
def ExistsManyGives(vars,cond,res):
    new_vars,var_map = get_new_vars(vars,"_ExistsManyGives")
    new_cond=substitute(cond,*var_map)
    new_res=substitute(res,*var_map)
    return Exists(vars+new_vars,And(new_cond,cond,new_res==res,Not(Equal(new_vars,vars))))

In [2]:
# Can you solve the rogue submarine riddle? - Alex Rosenthal
# https://youtu.be/iNgJCYPdmdQ?si=j0p8LbguRGwkUqZQ

s=SolverFor("LIA")
s=SolverFor("QF_LIA")
s=Solver()


a,b = Ints("a b")
sum,prod = Ints("sum prod")


common_conds = And(sum==a+b,prod==a*b,Distinct(a,b),1<=a,a<=6,1<=b,b<=6)


q1 = WithoutKnowingCanNotDetermine([a,b,prod], And(common_conds),  WithoutKnowingCanNotDetermine([a,b,sum], And(common_conds),  sum ) ) 
common_conds = And(common_conds,q1)
q2= WithoutKnowingCanDetermine([a,b,sum], And(common_conds),sum)
common_conds = And(common_conds,q2)

q3= WithoutKnowingCanDetermine([a,b,sum], And(common_conds),  WithoutKnowingCanDetermine([a,b,prod], And(common_conds),prod)) 
common_conds = And(common_conds,q3)


s.add(And(common_conds))




while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(sum),m.eval(prod))
        s.add(Not(And(m.eval(sum)==sum,m.eval(prod)==prod)))

    else:
        print(st)
        break

In [5]:
# Can you solve the secret sauce riddle? - Alex Gendler
# https://www.youtube.com/watch?v=HyRjuPP9S3o

s=SolverFor("QF_LIA")


s=Solver()
s=SolverFor("LIA")
SecondDigit = RecFunction("SecondDigit", IntSort(),IntSort())
n = FreshConst(IntSort())
RecAddDefinition(SecondDigit,[n],If(n<100, n/10, SecondDigit(n/10)))

a,b,t= Ints("a b t")


r1,r2,r3,r4 = Bools("r1 r2 r3 r4")
#s.add(a==64)#able to prove sat
s.add(a==729)#TODO not able to prove unsat
#s.add(a==728)#able to prove unsat

conds=[b!=a,13<=b,b<=1300,(b<500)==r1,Exists([t],t**2==b)==r2,Exists([t],t**3==b)==r3]
#not sure but from describtion of problem we can conclude that last question was asked to definitelly determine number, so before asking it there should be exactly 2 solutions
s.add(Only([a],And(13<=a,a<=1300,(a<500)==Not(r1),Exists([t],t**2==a)==Not(r2),Exists([t],t**3==a)==r3,(SecondDigit(a)==1)==Not(r4),ExistsN(2,[b],  And(conds)),ExistsOnly([b],  And(*conds,SecondDigit(b)==1)==r4))))
#TODO add condition that if we will answer questions correctly then we will get right number and only one number will satisfy to that conditions
while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(a))
        s.add(Not(And(m.eval(a)==a)))

    else:
        print(st)
        break

In [2]:
# Can you crack these 2 logical puzzles?
# https://youtu.be/ToIENnaMpLo?si=rR4BKlFOIB005E4N&t=55

s=SolverFor("LIA")
s=SolverFor("QF_LIA")
s=Solver()


a,b= Ints("a b")

common_conds = And(1<=a,a<=30,1<=b,b<=30)

#if A ask first question then he also does not know its answer
#but seems like this puzzle does not account it 
#p1 = WithoutKnowingCanNotDetermine([b], And(common_conds),  b==2*a ) ; common_conds = And(common_conds,p1)
#B also does not know but now he know that p1 is true
q1 = WithoutKnowingCanNotDetermine([a], And(common_conds),  b==2*a ) ; common_conds = And(common_conds,q1)
#p2 = WithoutKnowingCanNotDetermine([a], And(common_conds),  a==2*b ) ; common_conds = And(common_conds,p2)
q2 = WithoutKnowingCanNotDetermine([b], And(common_conds),  a==2*b ) ; common_conds = And(common_conds,q2)
#p3 = WithoutKnowingCanNotDetermine([b], And(common_conds),  a==2*b ) ; common_conds = And(common_conds,p3)
q3 = WithoutKnowingCanNotDetermine([a], And(common_conds),  a==2*b ) ; common_conds = And(common_conds,q3)
#p4 = WithoutKnowingCanNotDetermine([a], And(common_conds),  b==2*a ) ; common_conds = And(common_conds,p4)
q4 = WithoutKnowingCanNotDetermine([b], And(common_conds),  b==2*a ) ; common_conds = And(common_conds,q4)

q5 = WithoutKnowingCanDetermine([a], And(common_conds),  a ) ; common_conds = And(common_conds,q5)


s.add(common_conds)



while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(a))
        s.add(Not(And(m.eval(a)==a)))

    else:
        print(st)
        break

4
unsat


In [4]:
# An "unsolvable" logic puzzle
# https://youtu.be/8EITLJf1nug?si=k8hA7awtWmHJljyU

#TODO always return right answer but all solvers sometimes say unknown are there any other solution 

s=Solver()
s=SolverFor("LIA")
s=SolverFor("QF_LIA")

a,b,t1,t2= Ints("a b t1 t2")
s.add(OnlyGives([a,b],  And(0<=a,a<10,0<=b,b<10,a<b),   a*b%10))

while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(a+b))
        s.add(Not(And(m.eval(a+b)==(a+b))))

    else:
        print(st)
        break

10
unsat


In [7]:
# "UNSOLVABLE" Logic Puzzle: How Old Is The Priest?
# https://youtu.be/vhc4FceGjC8?si=2-DITEdee3QGWzYC


s=Solver()
s=SolverFor("QF_LIA")#solves faster


a,b,c,priest_age= Ints("a b c priest_age")
cond = And([a>0,b>0,c>0,a*b*c==2450,a<=b,b<=c])

#s.add(OneOfMany([a,b,c],  cond))#TODO returns unsat when I enable this condition
s.add(OneOfManyGives([a,b,c], And(cond,(a+b+c)%2==0),  a+b+c ))

s.add(OnlyGives([a,b,c], And(cond,(a+b+c)%2==0,a<priest_age,b<priest_age,c<priest_age),  a+b+c))

while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(a),m.eval(b),m.eval(c),m.eval(a+b+c),m.eval(priest_age))
        s.add(Not(And(m.eval(a)==a,m.eval(b)==b,m.eval(c)==c,m.eval(priest_age)==priest_age)))
    else:
        print(st)
        break


5 10 49 64 50
unsat


In [6]:
# Logic Puzzle of the age of three sons
# https://math.stackexchange.com/questions/40158/logic-puzzle-of-the-age-of-three-sons

# Can you solve the passcode riddle? - Ganesh Pai
# https://youtu.be/7Vd1dTBVbFg?si=MdB9dWRrElOqw3DW

s=Solver()


a,b,c= Ints("a b c")
cond = And([a>0,b>0,c>0,a*b*c==36,a<=b,b<=c])

s.add(OneOfManyGives([a,b,c], And(cond),  a+b+c ))#a,b,c is one many solutions that satisfy to cond and gives a+b+c result
s.add(OnlyGives([a,b,c], And(cond,a<c,b<c),  a+b+c ))

while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(a),m.eval(b),m.eval(c),m.eval(a+b+c))
        s.add(Not(And(m.eval(a)==a,m.eval(b)==b,m.eval(c)==c)))

    else:
        print(st)
        break


2 2 9 13
unsat
